# Luma Phase 2 Module Implementation
This notebook contain the implementation of phase 2 of Luma Geospatial Engine

## Prerequisite
Earth engine initialization using service account

In [1]:
import ee
ee.Authenticate()
ee.Initialize()

## Retrieve AOI from Earth Engine Asset Manager

In [2]:
from luma_ge.data_acquisition import GEE_Asset_Manager
#Initialize the Asset Manager from Earth Engine
asset = GEE_Asset_Manager()
if asset.load_asset():
    regency_names = asset.get_regency_names()
    if regency_names:
        print(f"✓ Found {len(regency_names)} regencies")
        #use city name
        selected_regency = "Kota Bandung" 
        #Verify the city name exists in the regency list
        if selected_regency in regency_names:
            print(f"\n✓ Selected regency: {selected_regency}")
        else:
            print(f"\n✗ Regency '{selected_regency}' not found")
            print(f"\nAvailable regencies:")
            for i, name in enumerate(regency_names, 1):
                print(f"  {i}. {name}")
            selected_regency = None
    else:
        print("✗ No regency names found")
        selected_regency = None
else:
    print("✗ Failed to load asset")
    selected_regency = None
#Retrieve AOI geometry for the selected regency
if selected_regency:
    aoi = asset.get_regency_geometry(selected_regency)
    
    if aoi:
        print(f"✓ Successfully retrieved geometry for: {selected_regency}")
    else:
        print(f"✗ Failed to retrieve geometry for: {selected_regency}")
        aoi = None
else:
    print("✗ No regency selected")
    aoi = None

2026-06-04 14:41:01,188 - luma_ge.data_acquisition - INFO - Successfully loaded asset with 548 features
2026-06-04 14:41:01,557 - luma_ge.data_acquisition - INFO - Extracted 516 unique regency names


✓ Found 516 regencies

✓ Selected regency: Kota Bandung


2026-06-04 14:41:02,922 - luma_ge.data_acquisition - INFO - Successfully retrieved geometry for: Kota Bandung


✓ Successfully retrieved geometry for: Kota Bandung


## New Feature: Sentinel-2 Data Retrieval



In [3]:
from luma_ge.data_acquisition import Reflectance_Data, final_Image
import geemap
#initialize class
optical_reflectance = Reflectance_Data()
composite = final_Image()
#define the temporal range
start = '2024-01-01'
end = '2024-12-30'
#retrieve the sentinel 2 data
s2_data, metas2 = optical_reflectance.get_s2_optical_data(aoi, start, end, cloud_cover=20, compute_detailed_stats=False)
#final output: composite image
median_s2 = composite.get_temporal_composite(s2_data, aoi, calculate_coverage=False, coverage_scale=10)
#sharpened composite image
sharpen_s2 = composite.get_temporal_composite(s2_data, aoi, sharpen = True)
#10m band
visnir = median_s2.select(['RED', 'GREEN', 'BLUE', 'NIR'])
#20m band
reswir = median_s2.select(['RED_EDGE1', 'RED_EDGE2', 'RED_EDGE3', 'RED_EDGE4', 'SWIR1', 'SWIR2'])
vis_10m = {'min': 0,'max': 0.3,'gamma': [0.95, 1.1, 1],'bands':['RED', 'GREEN', 'BLUE']}
res_20m = {'min': 0,'max': 0.3,'gamma': [0.95, 1.1, 1],'bands':['RED_EDGE1', 'RED_EDGE4', 'SWIR1']}
Map = geemap.Map()
Map.addLayer(median_s2, vis_10m, 'Sentinel-2 Composite 10 m band')
Map.addLayer(median_s2, res_20m, 'Sentinel-2 Composite 20 m band')
Map.centerObject(aoi, 10)
Map

2026-06-04 14:41:13,754 - luma_ge.ee_config - INFO - Earth Engine initialized successfully
2026-06-04 14:41:13,754 - Reflectance_Data - INFO - ReflectanceData initialized.
2026-06-04 14:41:13,755 - final_Image - INFO - final_Image creation initialized.
2026-06-04 14:41:13,755 - Reflectance_Data - INFO - Starting data fetch for Sentinel-2 Level-2A Surface Reflectance (Harmonized)
2026-06-04 14:41:13,756 - Reflectance_Data - INFO - Date range: 2024-01-01 to 2024-12-30
2026-06-04 14:41:13,756 - Reflectance_Data - INFO - Cloud cover threshold (image-level): 20%
2026-06-04 14:41:13,756 - Reflectance_Data - INFO - Cloud Score+ pixel threshold: 0.6
2026-06-04 14:41:13,757 - Reflectance_Data - INFO - Detailed statistics will not be computed
2026-06-04 14:41:13,757 - Reflectance_Stats - INFO - Reflectance Stats initialized.
2026-06-04 14:41:13,759 - Reflectance_Data - INFO - Filtered collection created (use compute_detailed_stats=True for more information)
2026-06-04 14:41:14,681 - final_Image 

Map(center=[-6.919241950180151, 107.63659926544979], controls=(WidgetControl(options=['position', 'transparent…

In [ ]:
from luma_ge.data_acquisition import Reflectance_Data, final_Image
import geemap
#initialize class
optical_reflectance = Reflectance_Data()
composite = final_Image()
#define the temporal range
start = '2024-01-01'
end = '2024-12-30'
#retrieve the sentinel 2 data
s2_data, metas2 = optical_reflectance.get_s2_optical_data(aoi, start, end, cloud_cover=20, compute_detailed_stats=False)
#final output: composite image
median_s2 = composite.get_temporal_composite(s2_data, aoi, calculate_coverage=False, coverage_scale=10)
Map = geemap.Map()
vis_10m = {'min': 0,'max': 5000,'gamma': [0.95, 1.1, 1],'bands':['RED', 'GREEN', 'BLUE']}
Map.addLayer(median_s2, vis_10m, 'Sentinel-2 Composite')
Map.addLayer(s2_data, vis_10m, 'Sentinel-2 Collection')
Map

In [ ]:
def create_panchromatic(
    image: ee.Image,
    crs: str = 'EPSG:32648',
    crs_transform: list = None,
) -> ee.Image:
    """
    Averages the four native 10 m bands (B2, B3, B4, B8) into a synthetic
    panchromatic band and reprojects to a fixed grid.

    Parameters
    ----------
    image : ee.Image
        Sentinel-2 composite.
    crs : str
        Target CRS (default: UTM Zone 48N for west Indonesia / mainland SE Asia).
    crs_transform : list[float], optional
        Affine transform [xScale, xShear, xOrig, yShear, yScale, yOrig].
        Defaults to 10 m grid anchored at (300000, 300000).

    Returns
    -------
    ee.Image  (single band)
    """
    if crs_transform is None:
        crs_transform = [10, 0, 300_000, 0, -10, 300_000]

    panchro = (
        image.select('BLUE')
             .add(image.select('RED'))
             .add(image.select('GREEN'))
             .add(image.select('NIR'))
             .divide(4)
    )
    return panchro.reproject(crs=crs, crsTransform=crs_transform)
def get_band_groups(
    image: ee.Image,
    crs: str = 'EPSG:32748',
) -> tuple[ee.Image, ee.Image]:
    """
    Splits a Sentinel-2 composite into its native 10 m and 20 m band groups,
    each reprojected to a consistent fixed grid.

    Parameters
    ----------
    image : ee.Image
        Sentinel-2 composite (all bands).
    crs : str
        Target CRS.

    Returns
    -------
    ten_m_bands : ee.Image   — B2, B3, B4, B8  (10 m grid)
    twenty_m_bands : ee.Image — B5, B6, B7, B8A, B11, B12  (20 m grid)
    """
    ten_m_transform    = [10,  0, 300_000,  0, -10,  300_000]
    twenty_m_transform = [20,  0, 300_000,  0, -20,  300_000]

    ten_m_bands = (
        image.select(['BLUE', 'RED', 'GREEN', 'NIR'])
             .reproject(crs=crs, crsTransform=ten_m_transform)
    )
    twenty_m_bands = (
        image.select(['RED_EDGE1', 'RED_EDGE2', 'RED_EDGE3', 'RED_EDGE4', 'SWIR1', 'SWIR2'])
             .reproject(crs=crs, crsTransform=twenty_m_transform)
    )
    return ten_m_bands, twenty_m_bands


In [ ]:
def hpf_sharpen(
    panchro: ee.Image,
    twenty_m_bands: ee.Image,
    aoi: ee.Geometry,
    mod: float = 0.25,
) -> ee.Image:
    """
    Sharpens 20 m Sentinel-2 bands to 10 m using the High Pass Filter (HPF)
    method (Gangkofner et al. 2008).

    Algorithm
    ---------
    1. Bilinear-resample 20 m bands to 10 m.
    2. Convolve the panchromatic image with a 5 × 5 high-pass kernel.
    3. Compute per-band injection weight:  W = σ(MS_band) / σ(HPF) × M
    4. Add weighted HPF detail to each resampled band:
       Pixel_out = Pixel_in + HPF × W

    Parameters
    ----------
    panchro : ee.Image
        Single-band synthetic panchromatic image (10 m).
    twenty_m_bands : ee.Image
        Original 20 m bands: B5, B6, B7, B8A, B11, B12.
    aoi : ee.Geometry
        Region used for standard-deviation estimation.
    mod : float
        Modulating factor controlling sharpening intensity (default 0.25).

    Returns
    -------
    ee.Image  (uint16)
        Six HPF-sharpened bands at 10 m, in the same band order as *twenty_m_bands*.
    """
    BANDS_20M = ['RED_EDGE1', 'RED_EDGE2', 'RED_EDGE3', 'RED_EDGE4', 'SWIR1', 'SWIR2']

    # 5a. Bilinear resample to 10 m  (preserves spatial frequency content)
    resampled_ms = twenty_m_bands.resample('bilinear').reproject(
        crs=twenty_m_bands.projection().crs(),
        scale=10,
    )

    # 5b. 5 × 5 high-pass kernel  (sum = 0 for a pure high-pass response)
    row        = [-1, -1,  -1, -1, -1]
    center_row = [-1, -1,  24, -1, -1]
    kernel = ee.Kernel.fixed(
        5, 5,
        [row, row, center_row, row, row],
        -3, -3,    # kernel origin (centre)
        False,
    )
    hpf = panchro.convolve(kernel)

    # 5c. Standard deviations for injection weighting
    #     Both reduceRegion calls are server-side; no getInfo() needed.
    def reduce_stddev(img):
        return ee.Dictionary(
            img.reduceRegion(
                reducer=ee.Reducer.stdDev(),
                geometry=aoi,
                bestEffort=True,
            )
        )

    ms_sd_dict  = reduce_stddev(resampled_ms)
    hpf_sd_dict = reduce_stddev(hpf)

    # HPF has a single band; grab its stddev as a scalar
    hpf_sd = ee.Number(ee.List(hpf_sd_dict.values()).get(0))

    # Per-band correction and addition
    # Python loop builds server-side lazy graph — no client computation here
    ms_sd_list = ms_sd_dict.values(BANDS_20M)   # ee.List, ordered

    sharpened = []
    for i, band in enumerate(BANDS_20M):
        ms_sd      = ee.Number(ms_sd_list.get(i))
        correction = hpf.multiply(ms_sd).divide(hpf_sd).multiply(mod)
        sharpened.append(resampled_ms.select(band).add(correction))

    return ee.Image.cat(sharpened).uint16()
def linear_stretch(
    ms_sharpened: ee.Image,
    twenty_m_bands: ee.Image,
    aoi: ee.Geometry,
) -> ee.Image:
    """
    Rescales *ms_sharpened* so that its per-band mean and standard deviation
    match those of the original *twenty_m_bands* (z-score histogram matching).

    Formula:
        output = (input − μ_sharp) / σ_sharp × σ_orig + μ_orig

    Parameters
    ----------
    ms_sharpened : ee.Image
        HPF-sharpened 20 m bands (uint16).
    twenty_m_bands : ee.Image
        Original 20 m bands used as the reference distribution.
    aoi : ee.Geometry
        Region used for statistics estimation.

    Returns
    -------
    ee.Image  (uint16)
    """
    BANDS_20M = ['RED_EDGE1', 'RED_EDGE2', 'RED_EDGE3', 'RED_EDGE4', 'SWIR1', 'SWIR2']

    def reduce(img, reducer):
        return ee.Dictionary(
            img.reduceRegion(reducer=reducer, geometry=aoi, bestEffort=True)
        )

    # Sharpened image statistics
    sharp_mean = _stats_to_image(reduce(ms_sharpened,   ee.Reducer.mean()),   BANDS_20M)
    sharp_sd   = _stats_to_image(reduce(ms_sharpened,   ee.Reducer.stdDev()), BANDS_20M)

    # Original 20 m reference statistics
    orig_mean  = _stats_to_image(reduce(twenty_m_bands, ee.Reducer.mean()),   BANDS_20M)
    orig_sd    = _stats_to_image(reduce(twenty_m_bands, ee.Reducer.stdDev()), BANDS_20M)

    return (
        ms_sharpened
        .subtract(sharp_mean)
        .divide(sharp_sd)
        .multiply(orig_sd)
        .add(orig_mean)
        .uint16()
    )


In [ ]:
def _stats_to_image(stats_dict: ee.Dictionary, bands: list[str]) -> ee.Image:
    """
    Converts a server-side ``ee.Dictionary`` of per-band statistics into a
    multi-band constant image (one constant band per entry).

    This avoids ``getInfo()`` calls and keeps all operations server-side.

    Parameters
    ----------
    stats_dict : ee.Dictionary
        Result of ``image.reduceRegion()``, keyed by band name.
    bands : list[str]
        Ordered band names to extract from *stats_dict*.

    Returns
    -------
    ee.Image
        Multi-band constant image renamed to *bands*.
    """
    # .values(bands) returns an ee.List in band order — fully server-side
    values = stats_dict.values(bands)
    imgs = [ee.Image.constant(values.get(i)) for i in range(len(bands))]
    return ee.Image.cat(imgs).rename(bands)


In [ ]:
panchro                    = create_panchromatic(median_s2, 'EPSG:32748')
ten_m_bands, twenty_m_bands = get_band_groups(median_s2, 'EPSG:32748')
ms_sharpened = hpf_sharpen(panchro, twenty_m_bands, aoi, 0.25)
ms_output = linear_stretch(ms_sharpened, twenty_m_bands, aoi)
res_20m = {'min': 0,'max': 0.3,'gamma': [0.95, 1.1, 1],'bands':['RED_EDGE1', 'RED_EDGE4', 'SWIR1']}
m = geemap.Map()
m.addLayer(ms_output, res_20m, 'REVIS')
m

In [ ]:
print(sharpen_s2.bandNames().getInfo())

In [ ]:
Map.addLayer(sharpen_s2, res_20m, 'Sharpened Sentinel-2 Composite 10 m band')
Map

In [ ]:
# Apply sharpening to the Sentinel-2 composite
sharpened_s2 = optical_reflectance.sharpen_s2_bands(median_s2, aoi=aoi)


if sharpened_s2 is not None:
    print("Sharpening applied successfully.")
    # Visualize sharpened RGB
    vis_sharp = {'min': 0, 'max': 0.3, 'gamma': [0.95, 1.1, 1], 'bands': ['RED_EDGE1', 'RED_EDGE2', 'RED_EDGE3']}
    Map.addLayer(sharpened_s2, vis_sharp, 'Sentinel-2 Sharpened RGB')
    Map.centerObject(aoi, 9)
    Map
else:
    print("Sharpening failed. Check that the input image has standardized Sentinel-2 band names.")
    

In [ ]:
# Sanity check — run this BEFORE calling sharpen_s2_bands
print(type(median_s2))                          # should be ee.Image
print(median_s2.bandNames().getInfo())          # should list your renamed bands
print(median_s2.select('NIR').projection().getInfo())  # should show a real CRS

In [ ]:
#print(sharpened_s2.bandNames().getInfo())
re1 = sharpened_s2.select('RED_EDGE1')
m = geemap.Map()
m.addLayer(sharpened_s2, res_20m, 'SharpenedBand')
m.addLayer(re1, {}, 'Red Edge Sharpen')
m

In [ ]:
print(type(sharpened_s2))   

In [4]:
export_band = sharpen_s2.select('RED_EDGE1','RED_EDGE2', 'RED_EDGE3', 'RED_EDGE4', 'SWIR1', 'SWIR2')
export_task = ee.batch.Export.image.toDrive(
     image=export_band,
     description='S2_SharpenLuma',
     folder='Earth Engine',
     fileNamePrefix='S2_SharpenLuma',
     scale=10,
     region=aoi,  # or aoi.geometry()
     maxPixels=1e13
 )
export_task.start()
import time

while export_task.active():
     print('Exporting... (status: {})'.format(export_task.status()['state']))
     time.sleep(10)

print('Export complete (status: {})'.format(export_task.status()['state']))

Exporting... (status: READY)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting.